In [ ]:
from sagemaker.huggingface.model import HuggingFaceModel
import sagemaker
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os

# Remove proxy vars if not needed
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

# Use the certifi-provided CA bundle
os.environ["AWS_CA_BUNDLE"] = "/Volumes/LaCie/Projects_portfolio/NLP/SupportIQ/venv/lib/python3.11/site-packages/certifi/cacert.pem"

# Test AWS call
import boto3
print(boto3.client('sts').get_caller_identity())

In [ ]:
sagemaker_session = sagemaker.Session()

In [ ]:
model_uri = 's3://sagemaker-us-east-1-720332985926/huggingface-pytorch-training-2025-07-05-23-42-49-059/output/model.tar.gz'

In [ ]:
huggingface_model = HuggingFaceModel(
    entry_point='batch_transform.py',
    model_data=model_uri,
    source_dir=os.getenv("SOURCE_DIR"),
    role=os.getenv("ROLE"),
    transformers_version='4.49.0',
    py_version='py312',
    pytorch_version='2.6.0',
    sagemaker_session=sagemaker_session
    )

### sagemaker endpoint 
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge'
    )


In [ ]:
#predictor.predict({"inputs": "Classify the intent:  Can I cacnel my ticket?"})


In [ ]:
transformer = huggingface_model.transformer(
    instance_count=1,
    instance_type='ml.g4dn.xlarge',
    strategy="SingleRecord",
    output_path='s3://sagemaker-us-east-1-720332985926/transformer/output/',
    accept="application/json",
    assemble_with="Line"
)

In [ ]:
transformer.transform(
    data="s3://gen-ai-repository/finetuning/flan-t5/data/roberta-test.jsonl",
    content_type="application/json",
    split_type='Line',
    wait=True	
)